# OSFT on DGX Spark: Orthogonal Subspace Fine-Tuning on Blackwell

**Algorithm:** OSFT (Orthogonal Subspace Fine-Tuning)  
**Model:** `ibm-granite/granite-3.3-2b-instruct`  
**Hardware:** NVIDIA DGX Spark (GB10, 130.7 GB unified memory)  
**Dataset:** `b-mc2/sql-create-context` (natural language → SQL)  

---

OSFT ([Nayak et al., 2025](https://arxiv.org/abs/2504.07097)) is a newer fine-tuning approach
that uses SVD decomposition to identify orthogonal subspaces for parameter updates. It can
reduce the amount of data needed for fine-tuning instruction-tuned models while maintaining
quality.

This is our **highest risk** experiment:
- OSFT uses FSDP (Fully Sharded Data Parallel) — running FSDP on a single GPU is unusual
- The `liger-kernel` dependency (for memory-efficient kernels) may not have ARM wheels
- SVD decomposition at training start adds computational overhead

The good news: FSDP is built into PyTorch (no extra install), and the memory estimate
(37–48 GB) should fit in our 130.7 GB of unified memory.

Let's see what happens.

## 1. DGX Spark System Profile

> **Estimated run time:** ~5 seconds

In [1]:
import platform
import subprocess
import torch
import os

print("=" * 60)
print("DGX Spark System Profile")
print("=" * 60)

print(f"\nPlatform:    {platform.machine()}")
print(f"Processor:   {platform.processor() or 'ARM (Grace)'}")
print(f"OS:          {platform.platform()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    cc = torch.cuda.get_device_capability(0)
    print(f"\nGPU:         {gpu_name}")
    print(f"GPU Memory:  {gpu_mem:.1f} GB")
    print(f"Compute Cap: {cc[0]}.{cc[1]}")
    print(f"CUDA:        {torch.version.cuda}")
else:
    print("\nNo CUDA GPU detected!")

print(f"\nPyTorch:     {torch.__version__}")
print(f"BF16:        {torch.cuda.is_bf16_supported()}")

total_ram = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024**3)
print(f"\nSystem RAM:  {total_ram:.1f} GB (shared with GPU on DGX Spark)")

DGX Spark System Profile

Platform:    aarch64
Processor:   aarch64
OS:          Linux-6.17.0-1008-nvidia-aarch64-with-glibc2.39

GPU:         NVIDIA GB10
GPU Memory:  121.7 GB
Compute Cap: 12.1
CUDA:        13.0

PyTorch:     2.10.0+cu130
BF16:        True

System RAM:  121.7 GB (shared with GPU on DGX Spark)


/home/frank/jupyterlab/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  queued_call()


## 2. Environment Check — FSDP & Liger Kernel

OSFT uses:
- **FSDP** (PyTorch native) — no install needed
- **mini-trainer** — the backend for OSFT in Training Hub
- **liger-kernel** (optional) — Triton-based fused kernels for memory efficiency

> **Estimated run time:** ~2 seconds

In [2]:
import importlib.metadata

packages = [
    "training-hub", "instructlab-training", "mini-trainer",
    "peft", "bitsandbytes", "trl", "accelerate",
    "transformers", "datasets", "torch",
    "liger-kernel", "triton",
]

print(f"{'Package':<25} {'Status':<15} {'Version'}")
print("-" * 55)
for pkg in packages:
    try:
        version = importlib.metadata.version(pkg)
        print(f"{pkg:<25} {'INSTALLED':<15} {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{pkg:<25} {'MISSING':<15} —")

Package                   Status          Version
-------------------------------------------------------
training-hub              INSTALLED       0.5.0
instructlab-training      INSTALLED       0.14.1
mini-trainer              MISSING         —
peft                      INSTALLED       0.18.1
bitsandbytes              INSTALLED       0.48.1
trl                       INSTALLED       0.19.1
accelerate                INSTALLED       1.10.1
transformers              INSTALLED       5.2.0
datasets                  INSTALLED       4.3.0
torch                     INSTALLED       2.10.0+cu130
liger-kernel              INSTALLED       0.7.0
triton                    INSTALLED       3.6.0


In [3]:
# Check FSDP availability (built into PyTorch)
try:
    from torch.distributed.fsdp import FullyShardedDataParallel
    print("FSDP: Available (PyTorch built-in)")
    FSDP_AVAILABLE = True
except ImportError:
    print("FSDP: NOT available — this is unexpected for PyTorch 2.x")
    FSDP_AVAILABLE = False

FSDP: Available (PyTorch built-in)


### 2a. Attempting to Install Liger Kernel

Liger kernel provides Triton-based fused kernels (RMSNorm, SwiGLU, CrossEntropy, etc.)
that can significantly reduce activation memory during training.

> **Expectation:** Liger uses Triton, which should work on any GPU that PyTorch supports.
> However, ARM + Blackwell may still cause issues.

> **Estimated run time:** 15–60 seconds (pip resolve + install)

In [4]:
%%time
!pip install liger-kernel 2>&1 | tail -15

CPU times: user 14.3 ms, sys: 12.1 ms, total: 26.5 ms
Wall time: 1.56 s


In [5]:
try:
    import liger_kernel
    print(f"liger-kernel installed! Version: {liger_kernel.__version__}")
    LIGER_AVAILABLE = True
except ImportError as e:
    print(f"liger-kernel not available: {e}")
    print("We'll run OSFT without Liger (use_liger=False).")
    LIGER_AVAILABLE = False
except Exception as e:
    print(f"liger-kernel import error: {e}")
    LIGER_AVAILABLE = False

liger-kernel import error: module 'liger_kernel' has no attribute '__version__'


## 3. Memory Estimation — OSFT on Granite 2B

OSFT memory usage is similar to SFT (it updates a subspace of all parameters, not a low-rank
adapter). The `unfreeze_rank_ratio` controls how much of the parameter space is unfrozen.

> **Estimated run time:** ~30–60 seconds (downloads model config from HuggingFace on first run)

In [6]:
%%time
from training_hub import estimate

GPU_MEM_BYTES = int(130.7 * 1024**3)
MODEL = "ibm-granite/granite-3.3-2b-instruct"

print("=" * 60)
print("Memory Estimate: OSFT — Granite 2B (rank ratio 0.25)")
print("=" * 60)
osft_est = estimate(
    training_method="osft",
    num_gpus=1,
    gpu_memory=GPU_MEM_BYTES,
    model_path=MODEL,
    unfreeze_rank_ratio=0.25,
    max_tokens_per_gpu=2048,
    use_liger=LIGER_AVAILABLE,
    verbose=2,
)

Memory Estimate: OSFT — Granite 2B (rank ratio 0.25)
CAUTION: This is a very rough estimate of OSFT's memory requirements.
Actual memory requirements may vary from the given estimate.
Estimations for ibm-granite/granite-3.3-2b-instruct:


Summary:
The expected amount of memory needed to run this model is about 37.9 GB
The lower and upper bounds are 34.5 - 44.8 GB
If you have 1 GPUs, you will need about 37.9 GB, with bounds of 34.5 - 44.8 GB per GPU


Component Breakdown:
Each GPU will need 9.4 GB to store the model parameters
Each GPU will need 18.9 GB to store the optimizer states
Each GPU will need 9.4 GB to store the gradients
Each GPU will need 0.6 GB to store the intermediate activations
Each GPU will need 1.0 GB to store the outputs
Up to 10.3 GB can be expected as overhead

Decision:
The proposed training setup should work for your hardware.

CPU times: user 2.79 s, sys: 225 ms, total: 3.02 s
Wall time: 1.78 s


In [19]:
def fmt_gb(b):
    return f"{b / 1024**3:.1f} GB"

print("\nMemory Summary:")
print(f"  Available:      {fmt_gb(GPU_MEM_BYTES)}")
print(f"  OSFT estimate:  {fmt_gb(osft_est[0])} – {fmt_gb(osft_est[2])}")
headroom = GPU_MEM_BYTES - osft_est[2]
print(f"  Headroom:       {fmt_gb(headroom)} (worst case)")
print(f"  Liger kernels:  {'Enabled' if LIGER_AVAILABLE else 'Disabled'}")
print(f"\nVerdict: {'Should fit.' if headroom > 0 else 'TIGHT — may OOM!'}")


Memory Summary:
  Available:      130.7 GB
  OSFT estimate:  34.5 GB – 44.8 GB
  Headroom:       85.9 GB (worst case)
  Liger kernels:  Disabled

Verdict: Should fit.


## 4. Dataset Preparation

Same dataset as the other notebooks — `sql-create-context` in JSONL messages format.

> **Estimated run time:** ~10–30 seconds (dataset download, or instant if cached from earlier notebooks)

In [7]:
%%time
from datasets import load_dataset
import json

ds = load_dataset("b-mc2/sql-create-context", split="train")
print(f"Dataset size: {len(ds)} examples")
print(f"Columns: {ds.column_names}")
print(f"\nSample:")
print(json.dumps(ds[0], indent=2))

Dataset size: 78577 examples
Columns: ['answer', 'question', 'context']

Sample:
{
  "answer": "SELECT COUNT(*) FROM head WHERE age > 56",
  "question": "How many heads of the departments are older than 56 ?",
  "context": "CREATE TABLE head (age INTEGER)"
}
CPU times: user 145 ms, sys: 4.74 ms, total: 150 ms
Wall time: 1.09 s


In [8]:
SUBSET_SIZE = 2000
DATA_PATH = "data/sql_train.jsonl"

os.makedirs("data", exist_ok=True)

system_prompt = (
    "You are a SQL expert. Given a natural language question and the relevant table schema, "
    "write the correct SQL query."
)

with open(DATA_PATH, "w") as f:
    for i, row in enumerate(ds):
        if i >= SUBSET_SIZE:
            break
        user_content = f"Question: {row['question']}\n\nContext:\n{row['context']}"
        record = {
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": row["answer"]},
            ]
        }
        f.write(json.dumps(record) + "\n")

print(f"Wrote {min(SUBSET_SIZE, len(ds))} examples to {DATA_PATH}")
print(f"File size: {os.path.getsize(DATA_PATH) / 1024:.1f} KB")

# Preview
with open(DATA_PATH) as f:
    sample = json.loads(f.readline())
print(f"\nPreview:")
print(json.dumps(sample, indent=2))

Wrote 2000 examples to data/sql_train.jsonl
File size: 1025.3 KB

Preview:
{
  "messages": [
    {
      "role": "system",
      "content": "You are a SQL expert. Given a natural language question and the relevant table schema, write the correct SQL query."
    },
    {
      "role": "user",
      "content": "Question: How many heads of the departments are older than 56 ?\n\nContext:\nCREATE TABLE head (age INTEGER)"
    },
    {
      "role": "assistant",
      "content": "SELECT COUNT(*) FROM head WHERE age > 56"
    }
  ]
}


## 5. Training Run — OSFT with Training Hub

OSFT has more required parameters than the other algorithms. Key settings:

- **`unfreeze_rank_ratio=0.25`**: Unfreezes 25% of the SVD-decomposed parameter space.
  Lower = less memory, higher = more capacity. 0.25 is a good starting point.
- **`use_liger`**: If liger-kernel installed, enables fused Triton kernels for memory savings.
- **`max_tokens_per_gpu`**: Conservative memory cap.
- **Backend:** `mini-trainer` (default for OSFT).

> **Estimated run time:** 15–45 minutes for OSFT on Granite 2B with 2000 examples
> (includes model download ~2 min if not cached, SVD decomposition ~2–5 min at init,
> data processing ~2 min, training ~10–35 min). The SVD decomposition is a one-time
> upfront cost unique to OSFT.

In [9]:
import time

CKPT_DIR = "checkpoints/osft_granite2b"
DATA_OUTPUT_DIR = "data/osft_processed"
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(DATA_OUTPUT_DIR, exist_ok=True)

print("Starting OSFT training...")
print(f"  Model:              {MODEL}")
print(f"  Data:               {DATA_PATH}")
print(f"  Checkpoint dir:     {CKPT_DIR}")
print(f"  Unfreeze rank:      0.25")
print(f"  Liger kernels:      {LIGER_AVAILABLE}")
print(f"  Backend:            mini-trainer")
print()

Starting OSFT training...
  Model:              ibm-granite/granite-3.3-2b-instruct
  Data:               data/sql_train.jsonl
  Checkpoint dir:     checkpoints/osft_granite2b
  Unfreeze rank:      0.25
  Liger kernels:      False
  Backend:            mini-trainer



In [10]:
from training_hub import osft

start_time = time.time()

try:
    result = osft(
        model_path=MODEL,
        data_path=os.path.abspath(DATA_PATH),
        ckpt_output_dir=os.path.abspath(CKPT_DIR),
        data_output_dir=os.path.abspath(DATA_OUTPUT_DIR),
        # OSFT-specific
        unfreeze_rank_ratio=0.25,
        use_liger=LIGER_AVAILABLE,
        # Training config
        effective_batch_size=8,
        max_tokens_per_gpu=2048,
        max_seq_len=512,
        learning_rate=2e-5,
        num_epochs=1,
        warmup_steps=10,
        lr_scheduler="cosine",
        # Single GPU
        nproc_per_node=1,
    )
    elapsed = time.time() - start_time
    print(f"\nOSFT training completed in {elapsed:.1f}s ({elapsed/60:.1f} min)")

except Exception as e:
    elapsed = time.time() - start_time
    print(f"\nOSFT training failed after {elapsed:.1f}s")
    print(f"Error: {type(e).__name__}: {e}")
    print("\nSee the Troubleshooting section below.")
    import traceback
    traceback.print_exc()

/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status


Generating train split: 0 examples [00:00, ? examples/s]

Ensuring dataset is compatible with legacy format. (num_proc=8):   0%|          | 0/2000 [00:00<?, ? examples/…

Converting samples into input_ids and labels... (num_proc=8):   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/2000 [00:00<?, ? examples/s]

...

2026-03-11 21:18:21,896 - mini_trainer.api_train - INFO - Starting training setup...
2026-03-11 21:18:21,896 - mini_trainer.api_train - INFO - Running training command as subprocess: torchrun --nnodes=1 --node-rank=0 --nproc-per-node=1 --rdzv-id=0 --standalone /home/frank/jupyterlab/.venv/lib/python3.12/site-packages/mini_trainer/train.py --model-name-or-path=ibm-granite/granite-3.3-2b-instruct --data-path=/home/frank/jupyterlab/rh/FrankTH/data/osft_processed/data.jsonl --batch-size=8 --max-tokens-per-gpu=2048 --learning-rate=2e-05 --num-warmup-steps=10 --lr-scheduler=cosine --lr-scheduler-kwargs={} --seed=42 --output-dir=/home/frank/jupyterlab/rh/FrankTH/checkpoints/osft_granite2b --training-mode=epoch --max-epochs=1 --max-steps=0 --max-tokens=0 --train-dtype=float32 --osft --osft-unfreeze-rank-ratio=0.25 --osft-upcast-dtype=float32 --save-final-checkpoint


Command: torchrun --nnodes=1 --node-rank=0 --nproc-per-node=1 --rdzv-id=0 --standalone /home/frank/jupyterlab/.venv/lib/python3.12/site-packages/mini_trainer/train.py --model-name-or-path=ibm-granite/granite-3.3-2b-instruct --data-path=/home/frank/jupyterlab/rh/FrankTH/data/osft_processed/data.jsonl --batch-size=8 --max-tokens-per-gpu=2048 --learning-rate=2e-05 --num-warmup-steps=10 --lr-scheduler=cosine --lr-scheduler-kwargs={} --seed=42 --output-dir=/home/frank/jupyterlab/rh/FrankTH/checkpoints/osft_granite2b --training-mode=epoch --max-epochs=1 --max-steps=0 --max-tokens=0 --train-dtype=float32 --osft --osft-unfreeze-rank-ratio=0.25 --osft-upcast-dtype=float32 --save-final-checkpoint
Logs will be saved to: /home/frank/jupyterlab/rh/FrankTH/checkpoints/osft_granite2b/training_log_node0.log
/home/frank/jupyterlab/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capabil

2026-03-11 21:18:27,298 - mini_trainer.api_train - ERROR - Training subprocess failed. Check logs for details.
2026-03-11 21:18:27,298 - mini_trainer.api_train - INFO - Waiting for process to exit (60s timeout)...



OSFT training failed after 10.3s
Error: RuntimeError: Training failed. Please check the logs at /home/frank/jupyterlab/rh/FrankTH/checkpoints/osft_granite2b/training_log_node0.log for details.

See the Troubleshooting section below.


Traceback (most recent call last):
  File "/tmp/ipykernel_36571/4112134570.py", line 6, in <module>
    result = osft(
             ^^^^^
  File "/home/frank/jupyterlab/.venv/lib/python3.12/site-packages/training_hub/algorithms/osft.py", line 564, in osft
    return algorithm.train(
           ^^^^^^^^^^^^^^^^
  File "/home/frank/jupyterlab/.venv/lib/python3.12/site-packages/training_hub/algorithms/osft.py", line 233, in train
    return self.backend.execute_training(all_params)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/frank/jupyterlab/.venv/lib/python3.12/site-packages/training_hub/algorithms/osft.py", line 443, in execute_training
    return run_training(
           ^^^^^^^^^^^^^
  File "/home/frank/jupyterlab/.venv/lib/python3.12/site-packages/mini_trainer/api_train.py", line 259, in run_training
    raise RuntimeError(
RuntimeError: Training failed. Please check the logs at /home/frank/jupyterlab/rh/FrankTH/checkpoints/osft_granite2b/training_log_node0.log

### 5a. Retry Without Liger (if first attempt failed)

If training failed with `use_liger=True`, it may be a Triton/Liger compatibility issue.
Let's try without it.

> **Estimated run time:** Same as above (15–45 minutes) if re-running from scratch

In [24]:
# Uncomment and run this cell if the training above failed with Liger-related errors

RETRY_WITHOUT_LIGER = False  # Set to True if needed

if RETRY_WITHOUT_LIGER:
    # Clear GPU memory
    torch.cuda.empty_cache()
    import gc; gc.collect()

    CKPT_DIR_NOLIGER = "checkpoints/osft_granite2b_noliger"
    os.makedirs(CKPT_DIR_NOLIGER, exist_ok=True)

    print("Retrying OSFT without Liger kernels...")
    start_time = time.time()

    try:
        result = osft(
            model_path=MODEL,
            data_path=os.path.abspath(DATA_PATH),
            ckpt_output_dir=os.path.abspath(CKPT_DIR_NOLIGER),
            data_output_dir=os.path.abspath(DATA_OUTPUT_DIR),
            unfreeze_rank_ratio=0.25,
            use_liger=False,  # Explicitly disabled
            effective_batch_size=8,
            max_tokens_per_gpu=2048,
            max_seq_len=512,
            learning_rate=2e-5,
            num_epochs=1,
            warmup_steps=10,
            lr_scheduler="cosine",
            nproc_per_node=1,
        )
        elapsed = time.time() - start_time
        print(f"\nOSFT (no Liger) completed in {elapsed:.1f}s ({elapsed/60:.1f} min)")
    except Exception as e:
        elapsed = time.time() - start_time
        print(f"\nOSFT (no Liger) also failed after {elapsed:.1f}s")
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()
else:
    print("No retry needed (or set RETRY_WITHOUT_LIGER=True to try).")

No retry needed (or set RETRY_WITHOUT_LIGER=True to try).


## 6. GPU Memory Usage

In [25]:
if torch.cuda.is_available():
    allocated = torch.cuda.max_memory_allocated() / 1024**3
    reserved = torch.cuda.max_memory_reserved() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3

    print("Peak GPU Memory Usage:")
    print(f"  Allocated: {allocated:.2f} GB")
    print(f"  Reserved:  {reserved:.2f} GB")
    print(f"  Total:     {total:.1f} GB")
    print(f"  Headroom:  {total - reserved:.1f} GB")
else:
    print("No CUDA device available for memory stats.")

Peak GPU Memory Usage:
  Allocated: 0.00 GB
  Reserved:  0.00 GB
  Total:     121.7 GB
  Headroom:  121.7 GB


## 7. Troubleshooting Log

### Known Issues on DGX Spark for OSFT

| Issue | Detail | Workaround |
|-------|--------|------------|
| FSDP single GPU | FSDP is designed for multi-GPU; single-GPU behavior may be suboptimal | Should still work — just no sharding benefit |
| liger-kernel ARM | Triton wheels may not exist for aarch64 | Run with `use_liger=False` |
| SVD overhead | SVD decomposition at init can be slow for large models | One-time cost, not per-step |
| Compute cap 12.1 | PyTorch warning about unsupported compute capability | Usually works despite warning |
| mini-trainer backend | Less battle-tested than instructlab-training | Document any unique errors |

### What Worked
- *Fill in after running*

### What Didn't Work
- *Fill in after running*

### Key Observations
- *Fill in after running*

## 8. Results & Next Steps

### Summary
- **Algorithm:** OSFT (Orthogonal Subspace Fine-Tuning)
- **Model:** Granite 2B
- **Hardware:** DGX Spark (GB10, 130.7 GB unified memory)
- **unfreeze_rank_ratio:** 0.25
- **Liger kernels:** *Enabled/Disabled — fill in*
- **Training time:** *Fill in after running*
- **Peak memory:** *Fill in after running*
- **Status:** *Fill in after running*

### Comparison Across All Three Methods

| Method | Memory Est. | Deps Required | Risk Level | Status |
|--------|------------|---------------|------------|--------|
| QLoRA (LoRA+SFT) | 2.8–3.9 GB | unsloth (optional) | Low | *Fill in* |
| Full SFT | 44–57 GB | DeepSpeed | Medium | *Fill in* |
| OSFT | 37–48 GB | liger-kernel (optional) | High | *Fill in* |

### Navigation
- [← Notebook 1: LoRA+SFT](01_lora_sft_spark.ipynb)
- [← Notebook 2: Full SFT](02_sft_spark.ipynb)